In [37]:
import pandas as pd
import numpy as np
import json
import os

In [38]:
bgg_data = pd.DataFrame()
# Load all JSON files from data/groups folder
data_files = []
for filename in os.listdir('data/groups'):
    if filename.endswith('.json'):
        with open(os.path.join('data/groups', filename), 'r', encoding='utf-8') as f:
            data_files.append(json.load(f))

bgg_data = pd.DataFrame([item for sublist in data_files for item in (sublist if isinstance(sublist, list) else [sublist])])

In [39]:
bgg_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 176818 entries, 0 to 176817
Data columns (total 30 columns):
 #   Column                Non-Null Count   Dtype 
---  ------                --------------   ----- 
 0   row_id                176818 non-null  str   
 1   type                  176818 non-null  str   
 2   boardgame             176818 non-null  str   
 3   description           176780 non-null  str   
 4   min_players           176818 non-null  str   
 5   max_players           176818 non-null  str   
 6   suggested_numplayers  176818 non-null  object
 7   min_playtime          176818 non-null  str   
 8   max_playtime          176818 non-null  str   
 9   playingtime           176818 non-null  str   
 10  minimum_age           176818 non-null  str   
 11  release_year          176818 non-null  str   
 12  average_rating        176818 non-null  str   
 13  num_of_ratings        176818 non-null  str   
 14  weight                176818 non-null  str   
 15  num_of_weights        176818

In [40]:
numericos = ['release_year', 'min_players', 'max_players', 'playingtime', 'minimum_age', 'num_of_ratings','weight', 
             'num_of_weights', 'average_rating', 'bayes_average', 'std_deviation','owned', 'trading', 'wanting', 'wishing']
for col in numericos:
    bgg_data[col] = pd.to_numeric(bgg_data[col], errors='coerce')
bgg_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 176818 entries, 0 to 176817
Data columns (total 30 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   row_id                176818 non-null  str    
 1   type                  176818 non-null  str    
 2   boardgame             176818 non-null  str    
 3   description           176780 non-null  str    
 4   min_players           176818 non-null  int64  
 5   max_players           176818 non-null  int64  
 6   suggested_numplayers  176818 non-null  object 
 7   min_playtime          176818 non-null  str    
 8   max_playtime          176818 non-null  str    
 9   playingtime           176818 non-null  int64  
 10  minimum_age           176818 non-null  int64  
 11  release_year          176818 non-null  int64  
 12  average_rating        176818 non-null  float64
 13  num_of_ratings        176818 non-null  int64  
 14  weight                176818 non-null  float64
 15  num_of_weig

In [41]:
# Guardamos el número de filas antes de aplicar los filtros para poder comparar después
filas_prefiltro = bgg_data.shape[0]

# Filtramos juegos con un número máximo de jugadores razonable (<30)
bgg_data = bgg_data[(bgg_data["max_players"]<=30)]

# Filtramos juegos con duración (playingtime) plausibles (<1440 minutos, es decir, menos de 24 horas) y mayor a 0
bgg_data = bgg_data[(bgg_data["playingtime"]<1440) & (bgg_data["playingtime"]>0)]

# Filtramos juegos con edad mínima razonable (<22 años)
bgg_data = bgg_data[(bgg_data["minimum_age"]<22)]

# Excluimos registros con años de lanzamiento futuros (<= 2026)
bgg_data = bgg_data[(bgg_data["release_year"]<=2026)]

# Exigimos un mínimo de valoraciones para mayor robustez (>30)
bgg_data = bgg_data[(bgg_data["num_of_ratings"]>30)]

# Excluimos juegos sin valoración promedio  y peso validos (>0)
bgg_data = bgg_data[(bgg_data["average_rating"]>0) & (bgg_data["weight"]>0)]

# Contamos las filas después de aplicar los filtros
filas_postfiltro = bgg_data.shape[0]

# Mostramos resumen del efecto de los filtros: antes, después, eliminadas y porcentaje eliminado
print(f"Filas antes del filtro: {filas_prefiltro}")
print(f"Filas después del filtro: {filas_postfiltro}")
print(f"Filas eliminadas: {filas_prefiltro - filas_postfiltro}")
print(f"Porcentaje de filas eliminadas: {((filas_prefiltro - filas_postfiltro) / filas_prefiltro) * 100:.2f}%")

Filas antes del filtro: 176818
Filas después del filtro: 35889
Filas eliminadas: 140929
Porcentaje de filas eliminadas: 79.70%


In [42]:
print(f"Juego más nuevo de la base de datos: {bgg_data.loc[bgg_data['release_year'].idxmax()]['boardgame']} = {bgg_data['release_year'].max()}")
print(f"Juego más antiguo de la base de datos: {bgg_data.loc[bgg_data['release_year'].idxmin()]['boardgame']} = {bgg_data['release_year'].min()}")

Juego más nuevo de la base de datos: Nippon: Zaibatsu = 2026
Juego más antiguo de la base de datos: Senet = -3500


Columnas que son object y que son candidatas a ser externalizadas en tablas aparte:

In [43]:
bgg_data[['suggested_numplayers','ranks', "categories", "mechanisms", "family", "designers", "artists","publishers"]] 

,suggested_numplayers,ranks,categories,mechanisms,family,designers,artists,publishers
0,"{'Best': '4', 'Recommended': '2', 'Not Recomme...","{'boardgame': '1', 'strategygames': '1'}","[Age of Reason, Economic, Industry / Manufactu...","[Chaining, End Game Bonuses, Hand Management, ...","[Cities: Birmingham (England), Components: Map...","[Gavan Brown, Matt Tolman, Martin Wallace]","[Gavan Brown, Lina Cossette, David Forest, Gui...","[Roxley, Arclight Games, Board Game Rookie, Bo..."
1,"{'Best': '2', 'Recommended': '3', 'Not Recomme...","{'boardgame': '2', 'strategygames': '2'}","[Animals, Card Game, Environmental]","[Contracts, End Game Bonuses, Events, Grid Cov...","[Animals: Okapi, Components: Hexagonal Tiles, ...",[Mathias Wigge],"[Steffen Bieker, Loïc Billiau, Dennis Lohausen]","[Feuerland Spiele, Capstone Games, CMON Global..."
2,"{'Best': '4', 'Recommended': '2', 'Not Recomme...","{'boardgame': '3', 'thematic': '1', 'strategyg...","[Environmental, Medical]","[Action Points, Cooperative Game, Hand Managem...","[Components: Map (Global Scale), Components: M...","[Rob Daviau, Matt Leacock]",[Chris Quilliams],"[Z-Man Games, Asterion Press, Devir, Filosofia..."
3,"{'Best': '3', 'Recommended': '2', 'Not Recomme...","{'boardgame': '4', 'thematic': '2', 'strategyg...","[Adventure, Exploration, Fantasy, Fighting, Mi...","[Action Queue, Action Retrieval, Campaign / Ba...","[Category: Dungeon Crawler, Components: Map (C...",[Isaac Childres],"[Alexandr Elichev, Lucile Mathieu, Josh T. McD...","[Cephalofair Games, Albi, Albi Polska, Arcligh..."
4,"{'Best': '4', 'Recommended': '3', 'Not Recomme...","{'boardgame': '5', 'strategygames': '4'}","[Movies / TV / Radio theme, Novel-based, Scien...","[Automatic Resource Growth, Card Play Conflict...","[Books: Dune, Game: Dune: Imperium, Misc: Long...",[Paul Dennen],"[Clay Brooks, Derek Herring, Raul Ramos, Nate ...","[Dire Wolf, Arclight Games, Broadway Toys LTD,..."
...,...,...,...,...,...,...,...,...
173133,"{'Best': '2', 'Recommended': '1', 'Not Recomme...",{'boardgame': 'Not Ranked'},"[Expansion for Base-game, Card Game, Comic Boo...","[Cooperative Game, Deck Construction, Hand Man...",[Collectible: Living Card Game (Fantasy Flight...,"[Michael Boggs, Caleb Grace]",[],[Fantasy Flight Games]
173134,"{'Best': '2', 'Recommended': '1', 'Not Recomme...",{'boardgame': 'Not Ranked'},"[Expansion for Base-game, Card Game, Comic Boo...","[Cooperative Game, Deck Construction, Hand Man...",[Collectible: Living Card Game (Fantasy Flight...,"[Michael Boggs, Caleb Grace]",[],[Fantasy Flight Games]
173256,"{'Best': '1', 'Recommended': '2', 'Not Recomme...",{'boardgame': 'Not Ranked'},"[Expansion for Base-game, Racing, Sports]",[],"[Admin: Upcoming Releases, Game: Heat – Pedal ...","[Asger Aleksandrov Granerud, Daniel Skjold Ped...",[Vincent Dutrait],"[Days of Wonder, Rebel Sp. z o.o.]"
173404,"{'Best': '3', 'Recommended': '2', 'Not Recomme...",{'boardgame': 'Not Ranked'},"[Expansion for Base-game, Card Game]","[Hand Management, Open Drafting, Race, Set Col...",[Theme: Vikings],[Thomas Dupont],[Antoine Carrion],[Bombyx]


Generamos tablas accesorias para eliminar las listas dentro de las columnas

In [44]:
game_designers = bgg_data[['row_id','designers']]
game_designers = game_designers.explode('designers')
bgg_data.drop(columns=['designers'], inplace=True)

game_artists = bgg_data[['row_id','artists']]
game_artists = game_artists.explode('artists')
bgg_data.drop(columns=['artists'], inplace=True)

game_mechanics = bgg_data[['row_id','mechanisms']]
game_mechanics = game_mechanics.explode('mechanisms')
bgg_data.drop(columns=['mechanisms'], inplace=True)


game_categories = bgg_data[['row_id','categories']]
game_categories = game_categories.explode('categories')
bgg_data.drop(columns=['categories'], inplace=True)

game_family = bgg_data[['row_id','family']]
game_family = game_family.explode('family')
bgg_data.drop(columns=['family'], inplace=True)

game_publishers = bgg_data[['row_id','publishers']]
game_publishers = game_publishers.explode('publishers')
bgg_data.drop(columns=['publishers'], inplace=True)
bgg_data.shape

(35889, 24)

En el caso de las columnas con diccionarios, el proceso es diferente pero obtenemos también las tablas accesorias

In [45]:
# Vectorized expansion usando json_normalize (mucho más rápido que apply(pd.Series))
suggested_players_expanded = pd.json_normalize(bgg_data['suggested_numplayers'])
suggested_numplayers_table = pd.concat([bgg_data[['row_id']].reset_index(drop=True), suggested_players_expanded.reset_index(drop=True)], axis=1)
bgg_data.drop(columns=['suggested_numplayers'], inplace=True)
print(suggested_numplayers_table.head(10))



   row_id Best Recommended Not Recommended
0  224517    4           2               1
1  342942    2           3              4+
2  161936    4           2              4+
3  174430    3           2              4+
4  397598    4           3               5
5  316554    3           2              4+
6  233078    6           5               1
7  115746    2           4               1
8  167791    3           2              5+
9  187645    2           4               1


In [46]:
# Ranks: normalizar y convertir a numérico por columnas de forma robusta
ranks_expanded = pd.json_normalize(bgg_data['ranks'])
# Unimos row_id con ranks normalizados
ranks_table = pd.concat([bgg_data[['row_id']].reset_index(drop=True), ranks_expanded.reset_index(drop=True)], axis=1)
ranks_table.sort_values(by='boardgame', inplace=True)
ranks_table.head(10)
bgg_data.drop(columns=['ranks'], inplace=True)


In [47]:
bgg_clean = bgg_data.copy()
print(bgg_clean.columns)


bgg_clean = bgg_clean.drop(columns=['min_playtime','max_playtime','description'])
bgg_clean = bgg_clean.merge(ranks_table[['row_id','boardgame']], on='row_id', how='left')
bgg_clean.rename(columns={'boardgame_y': 'rank_boardgame','boardgame_x': 'name'}, inplace=True)
bgg_clean.sort_values(by='rank_boardgame', inplace=True)
bgg_clean.tail(25)

bgg_expansions = bgg_clean[bgg_clean['type'] == 'boardgameexpansion'].copy()
bgg_games = bgg_clean[bgg_clean['type'] == 'boardgame'].copy()
bgg_clean

Index(['row_id', 'type', 'boardgame', 'description', 'min_players',
       'max_players', 'min_playtime', 'max_playtime', 'playingtime',
       'minimum_age', 'release_year', 'average_rating', 'num_of_ratings',
       'weight', 'num_of_weights', 'bayes_average', 'std_deviation',
       'language_dependency', 'owned', 'trading', 'wanting', 'wishing'],
      dtype='str')


,row_id,type,name,min_players,max_players,playingtime,minimum_age,release_year,average_rating,num_of_ratings,weight,num_of_weights,bayes_average,std_deviation,language_dependency,owned,trading,wanting,wishing,rank_boardgame
0,224517,boardgame,Brass: Birmingham,2,4,120,14,2018,8.56433,58318,3.8607,2850,8.39353,1.42790,1: No necessary in-game text,83744,367,1754,21874,1
9,187645,boardgame,Star Wars: Rebellion,2,4,240,14,2016,8.42185,36332,3.7535,1278,8.16701,1.36219,4: Extensive use of text - massive conversion ...,57374,393,1460,14975,10
99,201808,boardgame,Clank!: A Deck-Building Adventure,2,4,60,12,2016,7.76073,47406,2.2321,1090,7.59547,1.20797,3: Moderate in-game text - needs crib sheet or...,62182,615,1127,12544,100
990,56692,boardgame,Parade,2,6,45,8,2007,6.99327,8616,1.4571,315,6.65288,1.21890,1: No necessary in-game text,12831,170,294,1805,1000
9887,5235,boardgame,Samurai Blades: The Game of Man-to-Man Combat ...,2,6,240,14,1984,6.80140,179,2.3333,15,5.58889,1.35319,1: No necessary in-game text,545,20,21,47,10000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30562,149915,boardgameexpansion,Descent: Journeys in the Dark (Second Edition)...,2,5,90,14,2014,8.30001,1550,3.2571,35,6.51024,1.20856,4: Extensive use of text - massive conversion ...,7900,74,152,535,Not Ranked
30561,149903,boardgameexpansion,Among the Stars: Section Seal,2,6,30,12,2013,6.90375,80,2.5000,4,5.54742,1.17871,3: Moderate in-game text - needs crib sheet or...,497,38,21,37,Not Ranked
30560,149901,boardgameexpansion,Carcassonne: Südsee – Freitag,2,5,35,8,2013,6.96426,122,1.2857,7,5.56834,1.49914,1: No necessary in-game text,745,19,17,58,Not Ranked
30573,150029,boardgameexpansion,1911 Amundsen vs Scott: Expansion 4 – Food Depots,2,2,20,12,2013,6.72273,44,2.0000,4,5.51509,1.52597,1: No necessary in-game text,152,2,4,11,Not Ranked


In [48]:
# Export the main tables to CSV files
bgg_games.to_csv('data/source/bgg_games.csv', index=False)
bgg_expansions.to_csv('data/source/bgg_expansions.csv', index=False)
ranks_table.to_csv('data/source/ranks_table.csv', index=False)
suggested_numplayers_table.to_csv('data/source/suggested_numplayers_table.csv', index=False)
game_designers.to_csv('data/source/game_designers.csv', index=False)
game_artists.to_csv('data/source/game_artists.csv', index=False)
game_mechanics.to_csv('data/source/game_mechanics.csv', index=False)
game_categories.to_csv('data/source/game_categories.csv', index=False)
game_family.to_csv('data/source/game_family.csv', index=False)
game_publishers.to_csv('data/source/game_publishers.csv', index=False)

print("All tables exported successfully!")

All tables exported successfully!
